# Implementation of the model by Gu et al.

Steady-state redox model of photosynthetic linear electron transport.

## Model summary

The model provides a simplified steady-state description of linear photosynthetic electron transport. It represents electron flow through PSII, the plastoquinone pool, and cytochrome b$_6f$ using composite redox parameters rather than explicit elementary reaction steps.

The model is intended to link chlorophyll fluorescence-derived variables with estimates of photosynthetic electron transport rate, while remaining sufficiently compact for integration into broader photosynthesis or crop-scale modelling frameworks.

### Source:
https://doi.org/10.1111/pce.14563

In [ ]:
from mxlpy import Model
import math
import numpy as np
import matplotlib.pyplot as plt


def f_q(q: float, a_q: float) -> float:
    """Redox poise balance between Cyt b6f and PSII."""
    return (1.0 + a_q) / (1.0 + a_q * q)


def f_T(T: float, T0: float, E_T: float) -> float:
    """
    Placeholder standardized temperature response.
    Set E_T=0 or f_T=1 if temperature response is not fitted.
    """
    if E_T == 0:
        return 1.0
    return math.exp(E_T * (T - T0) / (T * T0))


def f_s(Jg: float, b_s: float, c_s: float) -> float:
    """
    Placeholder swelling/crowding function.
    Set b_s=0 or c_s=0 to collapse to 1.
    The exact functional form should be checked against the paper text.
    """
    return 1.0 / (1.0 + c_s * (1.0 - math.exp(-b_s * Jg)))


def j_psii_gu(
    q: float,
    U: float,
    R1: float,
    R2: float,
    q_r: float,
    a_q: float,
    b_s: float,
    c_s: float,
    Jg: float,
    T: float,
    T0: float,
    E_T: float,
) -> float:
    """
    Gu et al. steady-state redox PET model.

    J_PSII = 2 U f_T f_s f_q (q_r - q) q /
             [ (R1 + 2 R2 f_s f_q - 1) q + q_r ]
    """
    fq = f_q(q, a_q)
    fs = f_s(Jg, b_s, c_s)
    ft = f_T(T, T0, E_T)

    numerator = 2.0 * U * ft * fs * fq * (q_r - q) * q
    denominator = (R1 + 2.0 * R2 * fs * fq - 1.0) * q + q_r

    return numerator / denominator


def get_model() -> Model:
    return (
        Model()
        .add_variables(
            {
                # q is treated as externally supplied measured open PSII fraction
                "q": 0.7,
            }
        )
        .add_parameters(
            {
                # composite PET parameters
                "U": 250.0,  # µmol m-2 s-1, max PQ/PQH2 oxidation potential
                "R1": 0.2,  # first resistance
                "R2": 0.5,  # second resistance
                "q_r": 1.0,  # reversible PSII fraction
                # regulatory functions
                "a_q": 0.0,  # Cyt/PSII redox-poise parameter
                "b_s": 0.0,  # swelling/crowding parameter
                "c_s": 0.0,  # max crowding impact
                "Jg": 500.0,  # gross excitation flux
                # temperature response
                "T": 298.15,
                "T0": 298.15,
                "E_T": 0.0,
            }
        )
        .add_derived(
            "J_PSII",
            j_psii_gu,
            args=[
                "q",
                "U",
                "R1",
                "R2",
                "q_r",
                "a_q",
                "b_s",
                "c_s",
                "Jg",
                "T",
                "T0",
                "E_T",
            ],
        )
    )

In [ ]:
# Simplified light/temperature-independent version:
# f_T = 1, f_s = 1


def j_psii_gu_simple(
    q: float,
    U: float,
    R1: float,
    R2: float,
    q_r: float,
    a_q: float,
) -> float:
    fq = (1.0 + a_q) / (1.0 + a_q * q)
    return 2.0 * U * fq * (q_r - q) * q / ((R1 + 2.0 * R2 * fq - 1.0) * q + q_r)

## Reproduce Figure 3
### Redox stoichiometry relationship between Cyt and PSII.

h_cyt = fraction of cytochrome b6f complex available for LET

q     = fraction of open PSII reaction centres

a_q   = Cyt–PSII stoichiometry parameter

In [ ]:
def h_cyt(q, a_q):
    """
    Equation 14-style Cyt–PSII redox poise relation.

    h_cyt = q * (1 + a_q) / (1 + a_q * q)

    Valid domain:
    q in [0, 1]
    a_q > -1, otherwise h_cyt can become nonphysical.
    """
    return q * (1.0 + a_q) / (1.0 + a_q * q)

In [ ]:
q = np.linspace(0, 1, 500)

a_q_values = [10, 2, 0, -0.5, -0.9]

plt.figure(figsize=(6, 5))

for a_q in a_q_values:
    plt.plot(q, h_cyt(q, a_q), label=rf"$a_q={a_q:g}$")

plt.plot(q, q, linestyle="--", linewidth=1.2, label=r"$h_{cyt}=q$")

plt.xlabel(r"Fraction of open PSII reaction centres, $q$")
plt.ylabel(r"Fraction of Cyt available for LET, $h_{cyt}$")
plt.title("Cyt–PSII redox stoichiometry relationship")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## Reproduce Figure 8
Examples of the change in the oxidized fraction of mobile plastoquinone pool, the fraction of cytochrome b6f complex available for linear electron transport, and the fraction of open photosystem II reaction centres under the assumption of lake model as a function of photosynthetically active radiation (PAR).

Parameters available in the Supplementary Table 3.

In [ ]:
# ============================================================
# Parameters from uploaded supplementary table
# Used for the organism-specific OC-model reconstruction
# ============================================================

species_params = {
    "Solanum lycopersicum": {
        "U": 1714.1947,
        "R1": 0.3111,
        "R2": 0.0000,
        "q_r": 0.8598,
        "a_q": -0.8017,
        "b_s": 0.0042,
        "c_s": 15.5232,
    },
    "Betula alleghaniensis": {
        "U": 1189.0250,
        "R1": 0.4702,
        "R2": 0.0000,
        "q_r": 0.8636,
        "a_q": -0.6134,
        "b_s": 0.0014,
        "c_s": 6.8590,
    },
    "Liquidambar styraciflua": {
        "U": 846.8143,
        "R1": 0.5016,
        "R2": 0.0000,
        "q_r": 0.8906,
        "a_q": -0.5597,
        "b_s": 0.0020,
        "c_s": 4.6041,
    },
    "Oryza sativa": {
        "U": 963.4558,
        "R1": 0.5733,
        "R2": 0.0000,
        "q_r": 0.7623,
        "a_q": -0.5597,
        "b_s": 0.0024,
        "c_s": 7.2908,
    },
    "Carya ovata": {
        "U": 981.3335,
        "R1": 0.5668,
        "R2": 0.0000,
        "q_r": 0.7805,
        "a_q": -0.5548,
        "b_s": 0.0015,
        "c_s": 4.8562,
    },
    "Andropogon gerardii": {
        "U": 571.5778,
        "R1": 0.4538,
        "R2": 0.0000,
        "q_r": 0.9564,
        "a_q": -0.5711,
        "b_s": 0.0021,
        "c_s": 4.5994,
    },
    "Gossypium hirsutum": {
        "U": 1486.8670,
        "R1": 0.3988,
        "R2": 0.0011,
        "q_r": 0.9480,
        "a_q": -0.6342,
        "b_s": 0.0016,
        "c_s": 5.5211,
    },
    "Juglans nigra": {
        "U": 1312.6150,
        "R1": 0.4113,
        "R2": 0.0000,
        "q_r": 0.8110,
        "a_q": -0.7258,
        "b_s": 0.0021,
        "c_s": 5.5501,
    },
    "Zea mays": {
        "U": 726.9816,
        "R1": 0.3761,
        "R2": 0.0000,
        "q_r": 0.9196,
        "a_q": -0.6785,
        "b_s": 0.0021,
        "c_s": 5.5157,
    },
}

In [ ]:
def h_pq_visual_reconstruction(q_l, a_q):
    """
    Visual reconstruction of h_PQ.

    In the paper, h_PQ is obtained from the OC model, Eq. 28.
    Without the raw Table S1 measurement data and exact q_L values,
    this function reconstructs the visual relation in Figure 8:
    h_cyt < h_PQ < q_L for most points.

    It uses a weaker curvature than h_cyt.
    """
    effective_a = 0.35 * a_q
    return q_l * (1.0 + effective_a) / (1.0 + effective_a * q_l)


def q_l_from_par(PAR, q_min, q_max, k):
    """
    Empirical light-response decline of q_L with PAR.
    Used only to visually reconstruct the published figure.
    """
    return q_min + (q_max - q_min) * np.exp(-PAR / k)

In [ ]:
# ============================================================
# Species-specific PAR points and q_L response shapes
# Tuned to visually resemble Figure 8
# ============================================================

species_order = [
    "Solanum lycopersicum",
    "Betula alleghaniensis",
    "Liquidambar styraciflua",
    "Oryza sativa",
    "Carya ovata",
    "Andropogon gerardii",
    "Gossypium hirsutum",
    "Juglans nigra",
    "Zea mays",
]

panel_labels = ["a", "b", "c", "d", "e", "f", "g", "h", "i"]

shape_params = {
    "Solanum lycopersicum": {"q_min": 0.12, "q_max": 0.98, "k": 430},
    "Betula alleghaniensis": {"q_min": 0.22, "q_max": 0.98, "k": 520},
    "Liquidambar styraciflua": {"q_min": 0.25, "q_max": 0.98, "k": 620},
    "Oryza sativa": {"q_min": 0.10, "q_max": 0.95, "k": 330},
    "Carya ovata": {"q_min": 0.22, "q_max": 0.98, "k": 540},
    "Andropogon gerardii": {"q_min": 0.18, "q_max": 0.95, "k": 650},
    "Gossypium hirsutum": {"q_min": 0.32, "q_max": 0.98, "k": 750},
    "Juglans nigra": {"q_min": 0.20, "q_max": 0.98, "k": 500},
    "Zea mays": {"q_min": 0.28, "q_max": 0.98, "k": 600},
}

PAR_points = np.array([0, 40, 80, 120, 200, 300, 500, 700, 900, 1200, 1500, 1800])

In [ ]:
# ============================================================
# Plot Figure 8-style reconstruction
# ============================================================

fig, axes = plt.subplots(
    3,
    3,
    figsize=(8.2, 6.8),
    sharex=True,
    sharey=True,
)

axes = axes.ravel()

for ax, species, label in zip(axes, species_order, panel_labels):
    p = species_params[species]
    s = shape_params[species]

    q_l = q_l_from_par(
        PAR_points,
        q_min=s["q_min"],
        q_max=s["q_max"],
        k=s["k"],
    )

    h_cyt = h_cyt(q_l, p["a_q"])
    h_pq = h_pq_visual_reconstruction(q_l, p["a_q"])

    # keep all values in physical range
    q_l = np.clip(q_l, 0, 1)
    h_cyt = np.clip(h_cyt, 0, 1)
    h_pq = np.clip(h_pq, 0, 1)

    # h_PQ: triangle
    ax.plot(
        PAR_points,
        h_pq,
        marker="v",
        linestyle="None",
        markersize=4.5,
        markerfacecolor="black",
        markeredgecolor="black",
        label=r"$h_{PQ}$",
    )

    # h_cyt: solid dot
    ax.plot(
        PAR_points,
        h_cyt,
        marker="o",
        linestyle="None",
        markersize=4.2,
        markerfacecolor="black",
        markeredgecolor="black",
        label=r"$h_{cyt}$",
    )

    # q_L: open circle
    ax.plot(
        PAR_points,
        q_l,
        marker="o",
        linestyle="None",
        markersize=4.5,
        markerfacecolor="white",
        markeredgecolor="black",
        label=r"$q_L$",
    )

    ax.text(
        0.08,
        0.90,
        species,
        transform=ax.transAxes,
        fontsize=9,
        fontstyle="italic",
    )

    ax.text(
        0.05,
        0.08,
        f"({label})",
        transform=ax.transAxes,
        fontsize=9,
    )

    ax.set_xlim(0, 1850)
    ax.set_ylim(0, 1.0)

    ax.set_xticks([0, 400, 800, 1200, 1600])
    ax.set_yticks(np.linspace(0, 1.0, 6))

    ax.tick_params(
        direction="out",
        length=3,
        width=1,
        top=False,
        right=True,
        labelsize=8,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1.0)


# Axis labels
for ax in axes[6:9]:
    ax.set_xlabel(r"PAR ($\mu mol\ m^{-2}\ s^{-1}$)", fontsize=9)

for ax in axes[0::3]:
    ax.set_ylabel(r"$h_{PQ}$, $h_{cyt}$ or $q_L$ (unitless)", fontsize=9)


# Legend only in first panel, matching original style
axes[0].legend(
    frameon=False,
    fontsize=8,
    loc="upper right",
    handletextpad=0.4,
    borderpad=0.2,
)

fig.tight_layout(w_pad=0.25, h_pad=0.35)

plt.show()